# MangoLeaf AI — Training v3

**Trial 3: new dataset, scope-corrected.**

Same six classes. Same 70/15/15 split. Same MobileNetV3-Large. Same two-stage schedule
(5 epochs @ 1e-3, then 10 epochs @ 1e-5). Same calibration and rejection method as Trial 2.1.

What changes is the dataset. Trial 3 trains on `mango_dataset_v3`: the Trial 2.1 pool plus new
field photographs, minus images that fall outside the declared pre-harvest scope.

---

## What changed in the dataset

| | Trial 2.1 | Trial 3 |
| :-- | :--: | :--: |
| Raw images | 5,329 | 5,098 |
| Removed by perceptual hash | 2,567 | 2,495 |
| Unique images kept | 2,762 | 2,603 |
| Train / val / test | 1,930 / 415 / 417 | 1,819 / 391 / 393 |

Two changes produced that net decrease.

**Added.** New field photographs: a net gain of 8 in `anthracnose`, 2 in `healthy_leaf`, and 6
in `sooty_mold`. Each one survived deduplication one for one, so none of them duplicated an
image already in the pool.

**Removed by hand.** 247 files, 177 from `healthy_fruit` and 70 from `mango_scab`. These were
ripe or yellowing fruit. The app declares a pre-harvest scope, leaves and green fruit, so a
yellow ripe mango is out of scope by definition. Training `healthy` on ripe fruit teaches the
model that yellow means healthy, which is wrong at the point of use and blurs the boundary the
model actually has to draw: green healthy fruit against green diseased fruit.

Cross-checking against Trial 2.1's per-folder redundancy shows what kind of image left. Trial
2.1 recorded only 9 redundant images in the whole of `mango_scab`, so the 70 removals there took
out 66 distinct photographs rather than duplicates. In `healthy_fruit`, 109 of the 177 were
distinct.

## Class balance after the split

| Target class | Train | Val | Test | Total |
| :-- | :--: | :--: | :--: | :--: |
| `anthracnose` | 339 | 73 | 73 | 485 |
| `bacterial_black_spot` | 207 | 44 | 45 | 296 |
| `bacterial_canker` | 343 | 74 | 74 | 491 |
| `healthy` | 424 | 91 | 92 | 607 |
| `mango_scab` | 157 | 34 | 34 | 225 |
| `sooty_mold` | 349 | 75 | 75 | 499 |
| **TOTAL** | **1,819** | **391** | **393** | **2,603** |

Actual ratio 69.9% / 15.0% / 15.1%. No source image appears in more than one split.

The merged `healthy` class holds 497 leaf images and 110 fruit images. Fruit is 18% of healthy,
down from 31% in Trial 2.1. That is a direct consequence of the ripe-fruit exclusion and belongs
in Limitations: the healthy class is leaf-dominated, so a healthy green mango may come back less
confidently healthy than a healthy leaf does.

---

## What stays exactly the same

- Six output classes. **No seventh "not mango" class.**
- `healthy_leaf` and `healthy_fruit` merged into one `healthy` class.
- 70% / 15% / 15% split, stratified per class folder, `SEED = 42`.
- `PHASH_DISTANCE = 5`. Deduplication stays greedy and inside each source folder.
- MobileNetV3-Large, ImageNet weights, GlobalAveragePooling + 20% Dropout + Dense(6).
- Stage 1: 5 epochs, base frozen, Adam 1e-3. Stage 2: 10 epochs, BatchNormalization frozen,
  Adam 1e-5.
- The 70% confidence threshold. It is **fixed**, never tuned by this notebook.
- Temperature scaling, then margin and entropy thresholds fitted on out-of-scope images only.

---

## Expected outcome

Trial 2.1 measured 97.12% on 417 test images. Trial 3's test set holds 393, so one error moves
the headline figure by 0.25% instead of 0.24% and the confidence interval widens slightly.

The figure worth watching is not the headline. 11 of Trial 2.1's 12 test errors were
`bacterial_black_spot` against `mango_scab`. Trial 3 gives `mango_scab` 66 fewer unique images
than Trial 2.1 did, but every survivor is green. A class that no longer has to cover both ripe
and green presentations may separate better on fewer images. It may also do worse. This run is
what settles it.

Read the confusion matrix in Step 10 before anything else, and compare those two cells against
Trial 2.1's. If they improve, the exclusion was the right call and the manuscript can say so
with a measurement behind it rather than an argument.

---

## Known issues carried into this run

**49 cross-label near-duplicate pairs**, unchanged from Trial 2.1, almost all `anthracnose`
against `healthy`. The same image under two labels means one label is wrong, so nothing is
deleted automatically. **5 of the 49 landed in different splits**, which inflates test accuracy
slightly. Record the count in the trial report and inspect the pairs by hand.

**Small per-class test sets.** `mango_scab` has 34 test images and the fruit part of `healthy`
has 17. One error moves per-class recall by 2.9% and 5.9% respectively. Report raw counts
alongside every rate for those two, and do not quote four decimal places as though they were
precise.

**Rejection is improved, not solved.** A six-class softmax always distributes probability across
exactly those six classes. There is no output node meaning "none of the above", so some
out-of-scope images will still land on a disease with genuine-looking confidence. Quote the
measured rejection rate as a limitation rather than implying the problem is gone.

**Grad-CAM is diagnostic, not proof.** Heat sitting on lesions is reassuring. It does not
guarantee the model generalises to a Davao orchard, because the training photographs still come
mostly from Bangladesh and India.

## Step 0 — Environment

Run this cell first. If Colab asks to restart the runtime after the install, **restart it,
then run the notebook again from the top.** Installing `tensorflowjs` can change the
TensorFlow version, and you want the whole run on one version.

In [ ]:
!pip install -q tensorflowjs imagehash

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Step 1 — Configuration

Everything adjustable lives here. Nothing below this cell needs editing for a normal run.

In [ ]:
import os, shutil, json, random, math, hashlib, zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

# ---- reproducibility -------------------------------------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- paths -----------------------------------------------------------------
RAW_DIR   = "/content/drive/MyDrive/mango_dataset_v3"      # source images, 7 folders
OOD_DIR   = "/content/drive/MyDrive/mango_ood_test"     # non-mango photos (optional)
OUT_DIR   = "/content/dataset"                          # split output, local disk
EXPORT_DIR = "/content/export"                          # model artefacts
DRIVE_SAVE = "/content/drive/MyDrive/mangoleaf_v3"    # final zip destination

# ---- dataset layout --------------------------------------------------------
# Seven source folders collapse into six target classes: both healthy folders merge.
SOURCE_FOLDERS = [
    "anthracnose",
    "bacterial_black_spot",
    "bacterial_canker",
    "healthy_fruit",
    "healthy_leaf",
    "mango_scab",
    "sooty_mold",
]

def target_class(folder):
    return "healthy" if "healthy" in folder else folder

# Keras assigns label indices by sorted folder name. This must match CLASSES in
# src/lib/constants.ts exactly.
CLASS_NAMES = sorted({target_class(f) for f in SOURCE_FOLDERS})

# ---- split -----------------------------------------------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---- duplicate scan --------------------------------------------------------
PHASH_DISTANCE = 5      # Hamming distance (out of 64) counted as a near-duplicate
REMOVE_DUPLICATES = True

# ---- training --------------------------------------------------------------
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
STAGE1_EPOCHS, STAGE1_LR = 5,  1e-3
STAGE2_EPOCHS, STAGE2_LR = 10, 1e-5

# ---- augmentation (the six transforms named in the manuscript) -------------
AUG_ROTATION_DEG   = 30      # random rotation, up to 30 degrees
AUG_ZOOM           = 0.20    # zoom, up to 20%
AUG_BRIGHTNESS     = 0.20    # brightness adjustment
AUG_SHEAR          = 0.20    # shear transformation
AUG_FLIP           = "horizontal_and_vertical"   # horizontal flip + vertical flip

# ---- rejection -------------------------------------------------------------
CONFIDENCE_THRESHOLD = 0.70   # FIXED BY THE MANUSCRIPT. Never tuned.
MIN_ID_RETENTION     = 0.95   # keep at least 95% of correct mango predictions

# ---- export ----------------------------------------------------------------
QUANTIZE_FLOAT16 = False      # True halves file size, costs a little accuracy

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

print("Target classes (index order):")
for i, c in enumerate(CLASS_NAMES):
    print("  {} -> {}".format(i, c))

## Step 2 — Duplicate scan (report only)

These two cells **delete nothing.** They read every image and report what they find, so you
can see the scale of the problem before anything is removed.

### How it works

A perceptual hash (`phash`) is a 64-bit fingerprint of what an image *looks like*, not of
its bytes. Re-saving a JPEG changes every byte but barely moves the phash. Two images whose
fingerprints differ in only a few bits are near-duplicates.

Flips and rotations produce completely different fingerprints, so each image is hashed in
all 8 flip/rotate orientations and the smallest fingerprint is kept as its canonical one.
Two images that are mirror images of each other then collide exactly.

### Why it matters

The Roboflow-hosted Carabao Diseases Dataset contains generated augmented variants, and
MangoLeafBD contains several crops of the same physical leaves. A per-file random split
scatters those near-copies across train, validation, and test. The model then gets graded on
images it has effectively already seen.

Four classes scoring exactly 1.00 precision and 1.00 recall in Trial 2 is the symptom.
Telling anthracnose from bacterial canker on real field photographs is not a task where
perfect scores are plausible.

### Choosing the threshold

The second cell prints two tables before it decides anything.

The first shows how far each image sits from its nearest neighbour. The second shows how
many images would be removed at each candidate threshold.

Read the second table for a **cliff** — a sudden jump in the removal count between two
thresholds. Below the cliff you are removing copies of the same image. Above it you start
removing genuinely different photographs, which throws away real training data.

If `PHASH_DISTANCE` looks too aggressive against that table, change it in the Step 1 config
cell and rerun from the grouping cell down. The hashing cell does not need to run again.

### How images are grouped

Two deliberate choices, both about not over-deleting.

**Greedy, not transitive.** An image is dropped only when it is directly within the
threshold of an image already kept. Grouping by transitive chains instead — A near B, B near
C, C near D — lets a run of small steps merge images that are nowhere near each other. On a
dataset of visually similar leaves that chain can swallow an entire class.

**Per source folder, not across the whole pool.** A match between two folders means the same
image carries two different labels. That is a labelling error for you to resolve, not
something to settle by silently deleting one class's copy. Those pairs are listed at the end
of the report and left in place.

In [ ]:
import imagehash
from PIL import Image, ImageOps
from tqdm.auto import tqdm

def canonical_phash(path):
    # Hash all 8 dihedral orientations, keep the lexicographically smallest.
    # Mirror images and rotations then produce an identical canonical hash.
    im = Image.open(path).convert("L")
    variants = []
    for k in range(4):
        r = im.rotate(90 * k, expand=True)
        variants.append(r)
        variants.append(ImageOps.mirror(r))
    objs = [imagehash.phash(v) for v in variants]
    strs = [str(o) for o in objs]
    best = min(range(len(strs)), key=lambda i: strs[i])
    return objs[best]

records = []      # (path, source_folder, target_class)
for folder in SOURCE_FOLDERS:
    src = os.path.join(RAW_DIR, folder)
    if not os.path.isdir(src):
        print("MISSING FOLDER:", src)
        continue
    for name in sorted(os.listdir(src)):
        if name.lower().endswith(IMG_EXTS):
            records.append((os.path.join(src, name), folder, target_class(folder)))

print("Images found:", len(records))

bits, kept, corrupt = [], [], []
for path, folder, tgt in tqdm(records, desc="hashing"):
    try:
        h = canonical_phash(path)
        bits.append(h.hash.flatten().astype(np.int32))
        kept.append((path, folder, tgt))
    except Exception as e:
        corrupt.append((path, repr(e)))

print("Hashed: ", len(kept))
print("Corrupt:", len(corrupt))
for p, e in corrupt[:10]:
    print("  ", p, e)

In [ ]:
# Pairwise Hamming distance without building an N x N x 64 array.
# popcount(a XOR b) == sum(a) + sum(b) - 2 * dot(a, b) for 0/1 vectors.
B = np.stack(bits).astype(np.int32)          # (N, 64)
S = B.sum(axis=1)
D = S[:, None] + S[None, :] - 2 * (B @ B.T)  # (N, N) Hamming distances
np.fill_diagonal(D, 999)
N = len(kept)

# Upper-triangle mask. This has to be built explicitly: np.triu(D, 1) would set
# the lower triangle to 0, and 0 is below any distance threshold, so every
# lower-triangle entry would be reported as a duplicate pair.
UPPER = np.triu(np.ones((N, N), dtype=bool), 1)

# --- how close are these images to each other? ------------------------------
nn = D.min(axis=1)
print("Nearest-neighbour phash distance, {} images".format(N))
print("{:>10} {:>10} {:>12}".format("distance", "images", "cumulative"))
cum = 0
for d in range(0, 13):
    c = int((nn == d).sum())
    cum += c
    print("{:>10} {:>10} {:>12}".format(d, c, cum))
print("{:>10} {:>10} {:>12}".format(">12", int((nn > 12).sum()), N))
print("")
print("Distance 0 means an identical-looking image exists elsewhere in the set.")
print("Re-encoded or resized copies land around 0 to 3. Different photographs of")
print("the same physical leaf land around 4 to 10. Unrelated images sit above 15.")


# --- grouping ---------------------------------------------------------------
# Greedy representative selection, run inside each source folder.
#
# Greedy rather than transitive: an image is dropped only if it is directly
# within the threshold of an image already kept, so a chain of small steps
# cannot collapse a whole class into one group.
#
# Per folder rather than globally: a cross-folder match means the same image
# carries two different labels, which is a labelling error to fix by hand, not
# something to silently resolve by deleting one class's copy.
folder_idx = {}
for idx, (p, f, t) in enumerate(kept):
    folder_idx.setdefault(f, []).append(idx)


def dedupe(threshold):
    reps_all, assigned = [], {}
    for f in SOURCE_FOLDERS:
        idxs = sorted(folder_idx.get(f, []), key=lambda i: kept[i][0])
        reps = []
        for i in idxs:
            if reps:
                d = D[i, reps]
                j = int(np.argmin(d))
                if d[j] <= threshold:
                    assigned[i] = reps[j]
                    continue
            reps.append(i)
        reps_all.extend(reps)
    return sorted(reps_all), assigned


print("")
print("Removal at each threshold, deduplicated within each source folder:")
print("{:>10} {:>10} {:>10} {:>9}".format("threshold", "removed", "remaining", "percent"))
for t in (0, 1, 2, 3, 4, 5, 6, 8, 10, 12):
    reps_t, _ = dedupe(t)
    marker = "   <- PHASH_DISTANCE" if t == PHASH_DISTANCE else ""
    print("{:>10} {:>10} {:>10} {:>8.1f}%{}".format(
        t, N - len(reps_t), len(reps_t), 100.0 * (N - len(reps_t)) / N, marker))
print("")
print("Look for a cliff. A big jump between two thresholds is the boundary between")
print("copies of the same image and genuinely different photographs. If the number")
print("at PHASH_DISTANCE looks too aggressive, lower it in Step 1 and rerun from")
print("this cell down. Rerunning the hashing cell above is not necessary.")


survivors, assigned = dedupe(PHASH_DISTANCE)

# Rebuild groups for reporting: representative -> [representative, dropped...]
groups = {i: [i] for i in survivors}
for dropped, rep in assigned.items():
    groups[rep].append(dropped)
dup_groups = {r: m for r, m in groups.items() if len(m) > 1}
redundant = N - len(survivors)

print("")
print("=" * 64)
print("DUPLICATE SCAN REPORT")
print("=" * 64)
print("Total images            :", N)
print("Threshold applied       :", PHASH_DISTANCE)
print("Duplicate groups        :", len(dup_groups))
print("Redundant images        :", redundant)
print("Unique images remaining :", len(survivors))
print("Redundancy              : {:.1f}%".format(100.0 * redundant / N))
if redundant > 0.5 * N:
    print("")
    print("WARNING: more than half the dataset matched. That is high enough to be a")
    print("         threshold problem rather than a dataset problem. Check the cliff")
    print("         in the table above before trusting this.")

print("")
print("{:<24} {:>8} {:>10} {:>8}".format("source folder", "images", "redundant", "unique"))
for f in SOURCE_FOLDERS:
    idxs = folder_idx.get(f, [])
    if not idxs:
        continue
    surv = sum(1 for i in idxs if i in groups)
    print("{:<24} {:>8} {:>10} {:>8}".format(f, len(idxs), len(idxs) - surv, surv))

# --- cross-label collisions, reported and not deleted -----------------------
pair_mask = UPPER & (D <= PHASH_DISTANCE)
cross = []
for a, b in np.argwhere(pair_mask):
    a, b = int(a), int(b)
    if kept[a][2] != kept[b][2]:
        cross.append((sorted([kept[a][2], kept[b][2]]), [kept[a][0], kept[b][0]]))

print("")
print("Cross-label near-duplicate pairs:", len(cross))
for labels, paths in cross[:10]:
    print("  {} vs {}".format(labels[0], labels[1]))
    for p in paths:
        print("      ", os.path.basename(p))
if len(cross) > 10:
    print("  ... and {} more".format(len(cross) - 10))
if cross:
    print("")
    print("NOTE: the same image under two different labels means one label is wrong.")
    print("      Nothing is deleted for these. Inspect them by hand before the v3 run.")

print("")
print("Nothing has been deleted yet. The next cell applies the removal.")

## Step 3 — Split into 70 / 15 / 15

One representative image is kept from each duplicate group; the rest are dropped. The
survivors are then split.

Two details that matter:

**The split runs per source folder, not over the whole pool.** That is what makes it
stratified: every class keeps its 70/15/15 proportions, and `healthy_fruit` stays
represented in all three subsets instead of landing entirely in one of them by chance.

**The output directory is wiped first.** Trial 2 copied files into `dataset/` without
clearing it. Any leftover file from an earlier run stayed behind and silently joined the
next training run.

In [ ]:
from sklearn.model_selection import train_test_split

if REMOVE_DUPLICATES:
    chosen = list(survivors)
    print("Removing {} redundant images, keeping {}.".format(
        len(kept) - len(chosen), len(chosen)))
else:
    chosen = list(range(len(kept)))
    print("Duplicate removal DISABLED. Using all {} images.".format(len(chosen)))

pool = [kept[i] for i in chosen]

# Wipe stale output so nothing from a previous run survives.
shutil.rmtree(OUT_DIR, ignore_errors=True)
for split in ("train", "val", "test"):
    for c in CLASS_NAMES:
        os.makedirs(os.path.join(OUT_DIR, split, c), exist_ok=True)

manifest = {"train": [], "val": [], "test": []}

for folder in SOURCE_FOLDERS:
    files = sorted(p for (p, f, t) in pool if f == folder)
    if not files:
        print("{:<24} EMPTY, skipped".format(folder))
        continue
    if len(files) < 10:
        raise ValueError(
            "Folder {} has only {} images left after deduplication. A 70/15/15 split "
            "needs at least 10. Lower PHASH_DISTANCE in Step 1 and rerun from the "
            "grouping cell.".format(folder, len(files)))
    tgt = target_class(folder)

    train_f, temp_f = train_test_split(
        files, test_size=(VAL_FRAC + TEST_FRAC), random_state=SEED)
    val_f, test_f = train_test_split(
        temp_f, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), random_state=SEED)

    for split, group in (("train", train_f), ("val", val_f), ("test", test_f)):
        for src in group:
            # Prefix with the source folder so healthy_leaf and healthy_fruit files
            # cannot collide on filename inside the merged healthy folder.
            dst = os.path.join(OUT_DIR, split, tgt, folder + "__" + os.path.basename(src))
            shutil.copy(src, dst)
            manifest[split].append((src, dst, tgt))

    print("{:<24} train {:>5}   val {:>4}   test {:>4}".format(
        folder, len(train_f), len(val_f), len(test_f)))

In [ ]:
# Verification. If any of these assertions fires, do not train.
print("{:<24} {:>8} {:>8} {:>8} {:>8}".format("target class", "train", "val", "test", "total"))
totals = [0, 0, 0]
for c in CLASS_NAMES:
    n = []
    for k, split in enumerate(("train", "val", "test")):
        cnt = len(os.listdir(os.path.join(OUT_DIR, split, c)))
        n.append(cnt)
        totals[k] += cnt
    print("{:<24} {:>8} {:>8} {:>8} {:>8}".format(c, n[0], n[1], n[2], sum(n)))
print("{:<24} {:>8} {:>8} {:>8} {:>8}".format(
    "TOTAL", totals[0], totals[1], totals[2], sum(totals)))

grand = sum(totals)
print("")
print("Actual ratio: {:.1f}% / {:.1f}% / {:.1f}%".format(
    100 * totals[0] / grand, 100 * totals[1] / grand, 100 * totals[2] / grand))

# No source image may appear in more than one split.
split_of = {}
for split, entries in manifest.items():
    for src, dst, tgt in entries:
        assert src not in split_of, "LEAK: {} is in both {} and {}".format(
            src, split_of[src], split)
        split_of[src] = split
print("No source image appears in more than one split. OK")

# No surviving near-duplicate pair may straddle two splits. Within a folder the
# survivors are pairwise further apart than the threshold by construction, so
# this is really a check on cross-folder pairs, which deduplication leaves alone.
sub = D[np.ix_(chosen, chosen)]
m = np.triu(np.ones((len(chosen), len(chosen)), dtype=bool), 1) & (sub <= PHASH_DISTANCE)
straddle = []
for a, b in np.argwhere(m):
    pa, pb = kept[chosen[int(a)]][0], kept[chosen[int(b)]][0]
    if split_of.get(pa) != split_of.get(pb):
        straddle.append((split_of.get(pa), pa, split_of.get(pb), pb))

print("Near-duplicate pairs split across different subsets:", len(straddle))
for sa, pa, sb, pb in straddle[:10]:
    print("  {} {}".format(sa, os.path.basename(pa)))
    print("  {} {}".format(sb, os.path.basename(pb)))
if straddle:
    print("")
    print("These are cross-folder pairs, so they carry different labels and cannot be")
    print("deduplicated automatically. They inflate test accuracy slightly. Note the")
    print("count in TRIAL_3.md and inspect the pairs by hand.")
else:
    print("No near-duplicate leakage between train, validation, and test. OK")

In [ ]:
# Save the 70/15/15 split to Drive so the exact train / val / test membership can
# be reviewed by hand later. Two artefacts are written:
#
#   split_manifest.csv  one row per image: split, class, source folder, original
#                       path, and the path inside the split tree
#   dataset_split.zip   the split tree itself, so the images can be opened
#
# A zip is written instead of copying files one at a time because Drive charges
# per-file overhead; a few thousand small copies take far longer than one upload.
#
# This cell must run before the cell that reads model.json back, which reuses the
# name "manifest" for the TFJS weights manifest and would overwrite this one.

import csv

SPLIT_SAVE_DIR = os.path.join(DRIVE_SAVE, "dataset_split")
os.makedirs(SPLIT_SAVE_DIR, exist_ok=True)

csv_path = os.path.join(SPLIT_SAVE_DIR, "split_manifest.csv")
rows = 0
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["split", "target_class", "source_folder",
                     "file_name", "original_path", "split_path"])
    for split in ("train", "val", "test"):
        for src, dst, tgt in sorted(manifest[split], key=lambda r: (r[2], r[0])):
            source_folder = os.path.basename(os.path.dirname(src))
            writer.writerow([split, tgt, source_folder,
                             os.path.basename(src), src, dst])
            rows += 1

print("split_manifest.csv : {} rows".format(rows))
print("  {}".format(csv_path))

# Per-class counts, so the split can be read without opening the CSV.
print("")
print("{:<24}{:>8}{:>8}{:>8}{:>8}".format("class", "train", "val", "test", "total"))
for c in CLASS_NAMES:
    n = [sum(1 for (_, _, t) in manifest[s] if t == c)
         for s in ("train", "val", "test")]
    print("{:<24}{:>8}{:>8}{:>8}{:>8}".format(c, n[0], n[1], n[2], sum(n)))
tot = [len(manifest[s]) for s in ("train", "val", "test")]
print("{:<24}{:>8}{:>8}{:>8}{:>8}".format("ALL", tot[0], tot[1], tot[2], sum(tot)))

# Build the archive on local disk first, then copy it across. Writing a large
# archive straight into the Drive mount is slow and can stall part-way.
local_zip = shutil.make_archive("/content/dataset_split", "zip", OUT_DIR)
drive_zip = os.path.join(SPLIT_SAVE_DIR, "dataset_split.zip")
shutil.copy(local_zip, drive_zip)
print("")
print("dataset_split.zip  : {:.1f} MB".format(os.path.getsize(drive_zip) / 1e6))
print("  {}".format(drive_zip))


## Step 4 — Data loaders

`label_mode="int"` because `class_weight` requires integer labels.

`shuffle=False` on validation and test so predictions line up with labels row by row.
Without it the confusion matrix is meaningless.

No rescaling anywhere. MobileNetV3-Large contains its own `Rescaling` layer, so it expects
raw `[0, 255]` pixels. Dividing by 255 here would divide twice and crush every pixel to
near zero — that was the Trial 1 failure, 25.22% accuracy across 15 epochs.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUT_DIR, "train"),
    labels="inferred", label_mode="int", class_names=CLASS_NAMES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUT_DIR, "val"),
    labels="inferred", label_mode="int", class_names=CLASS_NAMES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(OUT_DIR, "test"),
    labels="inferred", label_mode="int", class_names=CLASS_NAMES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)

print("")
print("Class order:", train_raw.class_names)
assert train_raw.class_names == CLASS_NAMES
assert val_ds.class_names == CLASS_NAMES
assert test_ds.class_names == CLASS_NAMES
print("Class order matches CLASS_NAMES. Paste this exact order into src/lib/constants.ts.")

## Step 5 — Live augmentation, all six transforms

This is the fix for Trial 2's biggest omission. Trial 2 created an `ImageDataGenerator`,
used it to write duplicate files to disk, and then never passed it to `model.fit`. The
model trained on completely unaugmented images.

Here, augmentation runs **inside the input pipeline**, so every image arrives differently
transformed on every epoch. Over 15 epochs the model never sees the same exact pixels
twice.

### The six transforms

| Manuscript wording | Implementation |
| :-- | :-- |
| random rotation (up to 30 degrees) | `RandomRotation(30/360)` |
| horizontal flip | `RandomFlip("horizontal_and_vertical")` |
| vertical flip | same layer |
| brightness adjustment | `RandomBrightness(0.2, value_range=(0,255))` |
| zoom (up to 20%) | `RandomZoom(0.2)` |
| shear transformation | `RandomShear`, written below |

Keras has no version-stable `RandomShear`, so it is implemented directly with
`ImageProjectiveTransformV3`. The shear is applied about the image centre; without the
recentring terms the image would slide sideways out of frame instead of skewing in place.

### Why augmentation stays out of the model

The augmentation layers are applied with `dataset.map()`, not stacked inside the model.
Random operations placed inside the model get baked into the exported TensorFlow.js graph,
where they would keep firing at inference time and give a different answer every scan.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="mangoleaf")
class RandomShear(tf.keras.layers.Layer):
    # Random shear about the image centre. Keras ships no version-stable equivalent.
    def __init__(self, factor=0.2, fill_mode="NEAREST", **kwargs):
        super().__init__(**kwargs)
        self.factor = factor
        self.fill_mode = fill_mode

    def call(self, inputs, training=False):
        if not training:
            return inputs
        shape = tf.shape(inputs)
        n = shape[0]
        h = tf.cast(shape[1], tf.float32)
        w = tf.cast(shape[2], tf.float32)
        cx, cy = w / 2.0, h / 2.0

        sx = tf.random.uniform([n], -self.factor, self.factor)
        sy = tf.random.uniform([n], -self.factor, self.factor)
        one  = tf.ones_like(sx)
        zero = tf.zeros_like(sx)

        # Output-to-input map. a2 and b2 cancel the centre offset the shear introduces.
        transforms = tf.stack([one, sx, -sx * cy,
                               sy, one, -sy * cx,
                               zero, zero], axis=1)

        return tf.raw_ops.ImageProjectiveTransformV3(
            images=inputs,
            transforms=transforms,
            output_shape=shape[1:3],
            fill_value=0.0,
            interpolation="BILINEAR",
            fill_mode=self.fill_mode)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"factor": self.factor, "fill_mode": self.fill_mode})
        return cfg


augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(AUG_FLIP, seed=SEED),
    tf.keras.layers.RandomRotation(AUG_ROTATION_DEG / 360.0,
                                   fill_mode="nearest", seed=SEED),
    tf.keras.layers.RandomZoom(AUG_ZOOM, fill_mode="nearest", seed=SEED),
    RandomShear(AUG_SHEAR),
    tf.keras.layers.RandomBrightness(AUG_BRIGHTNESS, value_range=(0.0, 255.0), seed=SEED),
], name="augmentation")

print("Augmentation pipeline:")
for layer in augment.layers:
    print("  -", layer.name)

### Confirm augmentation actually fires

One image, eight random passes. If all eight panels look identical, augmentation is not
working and the rest of this notebook is pointless — stop and fix it here.

Trial 2 would have shown eight identical panels.

In [ ]:
sample_batch, _ = next(iter(train_raw))
one = tf.expand_dims(sample_batch[0], 0)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    out = augment(one, training=True)[0].numpy()
    ax.imshow(np.clip(out, 0, 255).astype("uint8"))
    ax.set_title("original" if i == 0 else "pass {}".format(i))
    ax.axis("off")
axes.flat[0].imshow(np.clip(one[0].numpy(), 0, 255).astype("uint8"))
plt.suptitle("Same source image, 8 independent augmentation passes", fontsize=13)
plt.tight_layout()
plt.show()

print("Pixel range after augmentation: {:.1f} to {:.1f}  (must stay near 0-255)".format(
    float(tf.reduce_min(out)), float(tf.reduce_max(out))))

In [ ]:
# Augment only the training stream. Validation and test stay untouched.
train_ds = (train_raw
            .map(lambda x, y: (augment(x, training=True), y),
                 num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

steps = tf.data.experimental.cardinality(train_raw).numpy()
print("Training batches per epoch:", steps)
print("Augmented image presentations across {} epochs: {:,}".format(
    STAGE1_EPOCHS + STAGE2_EPOCHS,
    int(steps) * BATCH_SIZE * (STAGE1_EPOCHS + STAGE2_EPOCHS)))

## Step 6 — Class weights

This replaces Trial 2's file duplication.

### The problem

The leaf classes have roughly 800 training images each. The fruit classes have about 210.
Left alone, the model can ignore fruit entirely and still score well overall, because
getting all the leaves right already gets it most of the way.

### Trial 2's approach, and why it backfired

Trial 2 wrote extra augmented copies of minority-class files to disk. The intent was
correct. The bug was that `healthy` was in the duplication list alongside the two minority
fruit classes:

```python
fruit_targets = ['bacterial_black_spot', 'mango_scab', 'healthy']   # healthy does not belong
```

`healthy` had already been merged from `healthy_leaf` + `healthy_fruit` to 1,047 images.
Tripling it produced roughly 3,141 — about 47% of the training set. The model learned that
guessing `healthy` is right nearly half the time, which is exactly the behaviour you saw:
photograph a shoe, get `Healthy`.

### This approach

`class_weight` scales each class's contribution to the loss instead of copying files.

```
weight = total_images / (num_classes * images_in_this_class)
```

A class with few images gets a weight above 1.0, so each of its mistakes costs the model
more. No files are created, so the 70/15/15 split stays exactly as measured, and no class
can be accidentally over-weighted the way `healthy` was.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = []
for c_idx, c in enumerate(CLASS_NAMES):
    n = len(os.listdir(os.path.join(OUT_DIR, "train", c)))
    train_labels.extend([c_idx] * n)
train_labels = np.array(train_labels)

weights = compute_class_weight("balanced",
                               classes=np.arange(len(CLASS_NAMES)),
                               y=train_labels)
class_weight = {i: float(w) for i, w in enumerate(weights)}

print("{:<24} {:>8} {:>10}".format("class", "images", "weight"))
for i, c in enumerate(CLASS_NAMES):
    print("{:<24} {:>8} {:>10.3f}".format(
        c, int((train_labels == i).sum()), class_weight[i]))
print("")
print("Above 1.0 means the class is under-represented and its errors are penalised more.")

## Step 7 — Model

Identical architecture to Trial 2, with one structural change: the final layer is split in
two.

Trial 2 had:

```python
outputs = Dense(6, activation='softmax')(x)
```

This notebook has:

```python
logits  = Dense(6, name='logits')(x)
outputs = Activation('softmax', name='probs')(logits)
```

Mathematically identical. But the raw pre-softmax values (the logits) are now reachable as
a named layer, which temperature scaling in Step 11 needs. Softmax squashes everything into
0-1 and discards the scale information that calibration works on.

In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet")

base_model.trainable = False

inputs  = tf.keras.Input(shape=IMG_SIZE + (3,), name="input_image")
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
x       = tf.keras.layers.Dropout(0.2, name="dropout")(x)
logits  = tf.keras.layers.Dense(len(CLASS_NAMES), name="logits")(x)
outputs = tf.keras.layers.Activation("softmax", dtype="float32", name="probs")(logits)

model = tf.keras.Model(inputs, outputs, name="mangoleaf_v3")

print("Base layers      :", len(base_model.layers))
print("Trainable weights:", len(model.trainable_weights), "(stage 1: head only)")
model.summary()

## Step 8 — Stage 1: train the head

5 epochs, base frozen, Adam at 1e-3. Same as Trial 2, plus two additions:

**`class_weight`** — applied from this stage on.

**`ModelCheckpoint(save_best_only=True)`** — Trial 2 had no checkpointing, so it exported
whatever epoch 10 happened to produce. Its logs show epoch 9 had the better validation loss
(0.0495 against 0.0511). It shipped the worse model. This saves the best weights instead.

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)
CKPT_S1 = os.path.join(EXPORT_DIR, "stage1_best.weights.h5")
CKPT_S2 = os.path.join(EXPORT_DIR, "stage2_best.weights.h5")

model.compile(
    optimizer=tf.keras.optimizers.Adam(STAGE1_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.ModelCheckpoint(
        CKPT_S1, monitor="val_loss", mode="min",
        save_best_only=True, save_weights_only=True, verbose=1)])

model.load_weights(CKPT_S1)
print("")
print("Stage 1 complete. Best-validation weights restored.")

## Step 9 — Stage 2: fine-tune, BatchNormalization frozen

10 epochs, Adam at 1e-5. Same schedule as Trial 2, with one important difference.

### The BatchNormalization problem

Trial 2 did this:

```python
base_model.trainable = True
```

and accuracy fell from **0.9735 to 0.8535 in a single epoch.**

That drop is not the learning rate. Every BatchNormalization layer in MobileNetV3-Large
carries running mean and variance statistics learned from 1.4 million ImageNet images.
Setting `trainable = True` lets those statistics be overwritten by batches of 32 mango
photographs. The network has to relearn its own internal normalisation from scratch, and
every layer downstream sees shifting inputs while it does.

It recovered by epoch 10, but it spent the run repairing self-inflicted damage instead of
learning about mango disease.

### The fix

Unfreeze the convolutional weights, keep the BatchNormalization layers frozen:

```python
base_model.trainable = True
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
```

This is also what makes the manuscript's phrase "fine-tuning with **select** base layers
unfrozen" (line 265) literally true. Trial 2 unfroze everything.

In [ ]:
base_model.trainable = True

frozen_bn = 0
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        frozen_bn += 1

print("BatchNormalization layers frozen:", frozen_bn)
print("Trainable weight tensors        :", len(model.trainable_weights))

# Recompile so the trainable-flag changes take effect.
model.compile(
    optimizer=tf.keras.optimizers.Adam(STAGE2_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_EPOCHS,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.ModelCheckpoint(
        CKPT_S2, monitor="val_loss", mode="min",
        save_best_only=True, save_weights_only=True, verbose=1)])

model.load_weights(CKPT_S2)
print("")
print("Stage 2 complete. Best-validation weights restored.")
print("")
print("Compare epoch 1 accuracy against the end of stage 1. Trial 2 dropped 0.9735 to")
print("0.8535 here. With BatchNormalization frozen it should stay close instead.")

## Step 10 — Evaluate on the held-out test set

This produces the numbers for the manuscript's classification report table.

The baseline to beat is Trial 2.1: **97.12%**, 405 of 417 correct. Trial 3's test set holds
393 images, so anything from roughly 95% to 98% is the same performance within sampling
noise. Do not read a half-percent move as an improvement.

Two things to check before the headline number. Whether four classes still show a perfect
1.00 F1, which would mean duplicates remain and `PHASH_DISTANCE` needs loosening. And
whether the `bacterial_black_spot` / `mango_scab` confusion improved: 11 of Trial 2.1's 12
errors were that one pairing, and the ripe-fruit exclusion was supposed to help it.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print("Test accuracy: {:.4f}   ({:.2f}%)".format(test_acc, test_acc * 100))
print("Test loss    : {:.4f}".format(test_loss))
print("")
print("Trial 2.1 reported 97.12% on 417 test images. Trial 3 has 393.")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

print("")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Trial 3 confusion matrix — test set")
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
fig.colorbar(im)
plt.tight_layout()
plt.show()

print("Most-confused class pairs:")
off = [(cm[i, j], CLASS_NAMES[i], CLASS_NAMES[j])
       for i in range(len(CLASS_NAMES)) for j in range(len(CLASS_NAMES)) if i != j]
for n, a, b in sorted(off, reverse=True)[:6]:
    if n:
        print("  {:>4}  actual {} predicted as {}".format(n, a, b))

In [ ]:
acc  = hist1.history["accuracy"]     + hist2.history["accuracy"]
vacc = hist1.history["val_accuracy"] + hist2.history["val_accuracy"]
loss = hist1.history["loss"]         + hist2.history["loss"]
vloss= hist1.history["val_loss"]     + hist2.history["val_loss"]
cut  = len(hist1.history["accuracy"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(acc, label="train")
a1.plot(vacc, label="validation")
a1.axvline(cut - 0.5, ls="--", c="grey")
a1.text(cut - 0.4, min(acc), " stage 2 begins", fontsize=9, color="grey")
a1.set_title("Accuracy"); a1.set_xlabel("epoch"); a1.legend(); a1.grid(alpha=0.3)

a2.plot(loss, label="train")
a2.plot(vloss, label="validation")
a2.axvline(cut - 0.5, ls="--", c="grey")
a2.set_title("Loss"); a2.set_xlabel("epoch"); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Best validation loss: {:.4f} at epoch {}".format(min(vloss), int(np.argmin(vloss)) + 1))
print("Final epoch validation loss: {:.4f}".format(vloss[-1]))
if np.argmin(vloss) != len(vloss) - 1:
    print("The final epoch was NOT the best. ModelCheckpoint restored the better weights.")
    print("Trial 2 had no checkpoint and shipped the final epoch regardless.")

## Step 10b — Figures and tables for the manuscript

Everything above draws on screen only. Close the Colab tab and it is gone. This section
writes the same results to image and spreadsheet files at 200 DPI, sized to sit in a Word
document at one-column or full-page width.

### What gets written

| File | What it is | Where it goes in the manuscript |
| :-- | :-- | :-- |
| `confusion_matrix.png` | Two panels: raw counts, and each row as percentages | The confusion matrix named in the Evaluation Metrics section |
| `per_class_metrics.png` | Grouped bars: precision, recall, F1 per class | Visual companion to the classification report |
| `per_class_metrics.csv` | The same numbers as a table | Paste into Word, format as a table |
| `training_curves.png` | Accuracy and loss across all 15 epochs | Training and fine-tuning discussion |
| `grad_cam.png` | Heatmaps over one test image per class | Model interpretability, if you include it |
| `rejection_distributions.png` | Confidence, margin, and entropy histograms | Only produced when the out-of-scope folder exists |

### Reading the two confusion-matrix panels

The **left panel** is raw image counts. Use it to state exact numbers.

The **right panel** divides each row by that row's total, so every row sums to 100%. That is
the one to read during defense, because your classes have very different sizes — 8 errors
out of 40 fruit images is a much worse result than 8 errors out of 400 leaf images, and only
the percentage panel makes that visible.

The diagonal of the right panel is per-class recall. Anything off-diagonal is a confusion
worth being able to explain out loud.

In [ ]:
import csv

FIG_DIR = os.path.join(EXPORT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DPI = 200

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
})

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                               digits=4, output_dict=True)

# --- Figure: confusion matrix, counts beside row percentages ----------------
row_totals = np.maximum(cm.sum(axis=1, keepdims=True), 1)
cm_pct = cm.astype(np.float64) / row_totals * 100.0

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
panels = [
    (axes[0], cm, "Blues", "Counts", "{:d}", None),
    (axes[1], cm_pct, "Blues", "Row percentages (per-class recall on diagonal)",
     "{:.1f}", 100.0),
]
for ax, data, cmap, title, fmt, vmax in panels:
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax if vmax else data.max())
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_title(title)
    limit = (vmax if vmax else data.max()) / 2.0
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            txt = fmt.format(int(v) if fmt == "{:d}" else v)
            if fmt != "{:d}":
                txt += "%"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                    color="white" if v > limit else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Confusion matrix on the held-out test set (n = {}), Trial 3".format(
    len(y_true)), fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=FIG_DPI)
plt.show()


# --- Figure: per-class precision, recall, F1 --------------------------------
prec = [report[c]["precision"] for c in CLASS_NAMES]
rec  = [report[c]["recall"] for c in CLASS_NAMES]
f1   = [report[c]["f1-score"] for c in CLASS_NAMES]
sup  = [int(report[c]["support"]) for c in CLASS_NAMES]

x = np.arange(len(CLASS_NAMES))
w = 0.26

fig, ax = plt.subplots(figsize=(12, 5.5))
for offset, vals, label in [(-w, prec, "Precision"), (0, rec, "Recall"), (w, f1, "F1-score")]:
    bars = ax.bar(x + offset, vals, w, label=label)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.012, "{:.2f}".format(v),
                ha="center", va="bottom", fontsize=8)

ax.axhline(report["accuracy"], ls="--", c="grey", lw=1,
           label="Overall accuracy {:.2%}".format(report["accuracy"]))
ax.set_xticks(x)
ax.set_xticklabels(["{}\n(n={})".format(c, s) for c, s in zip(CLASS_NAMES, sup)],
                   fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Per-class precision, recall, and F1-score on the test set, Trial 3")
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "per_class_metrics.png"), dpi=FIG_DPI)
plt.show()


# --- Table: same numbers as CSV, plus the averages --------------------------
csv_path = os.path.join(FIG_DIR, "per_class_metrics.csv")
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["Class", "Precision", "Recall", "F1-score", "Support"])
    for c in CLASS_NAMES:
        r = report[c]
        wr.writerow([c, "{:.4f}".format(r["precision"]), "{:.4f}".format(r["recall"]),
                     "{:.4f}".format(r["f1-score"]), int(r["support"])])
    wr.writerow([])
    wr.writerow(["Overall accuracy", "{:.4f}".format(report["accuracy"]), "", "",
                 len(y_true)])
    for avg in ("macro avg", "weighted avg"):
        r = report[avg]
        wr.writerow([avg.title(), "{:.4f}".format(r["precision"]),
                     "{:.4f}".format(r["recall"]), "{:.4f}".format(r["f1-score"]),
                     int(r["support"])])

# Same table on screen, spaced for a direct paste into Word.
print("{:<24} {:>10} {:>10} {:>10} {:>9}".format(
    "Class", "Precision", "Recall", "F1-score", "Support"))
print("-" * 66)
for c in CLASS_NAMES:
    r = report[c]
    print("{:<24} {:>10.4f} {:>10.4f} {:>10.4f} {:>9d}".format(
        c, r["precision"], r["recall"], r["f1-score"], int(r["support"])))
print("-" * 66)
print("{:<24} {:>10.4f} {:>10} {:>10} {:>9d}".format(
    "Overall accuracy", report["accuracy"], "", "", len(y_true)))
for avg in ("macro avg", "weighted avg"):
    r = report[avg]
    print("{:<24} {:>10.4f} {:>10.4f} {:>10.4f} {:>9d}".format(
        avg.title(), r["precision"], r["recall"], r["f1-score"], int(r["support"])))


# --- Figure: training curves ------------------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tr, va, name in [(a1, acc, vacc, "Accuracy"), (a2, loss, vloss, "Loss")]:
    ax.plot(range(1, len(tr) + 1), tr, marker="o", ms=3, label="Training")
    ax.plot(range(1, len(va) + 1), va, marker="s", ms=3, label="Validation")
    ax.axvline(cut + 0.5, ls="--", c="grey", lw=1)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(name)
    ax.set_title("{} — stage 1 (frozen base) then stage 2 (fine-tuning)".format(name),
                 fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
a2.annotate("stage 2 begins", xy=(cut + 0.5, max(vloss)),
            xytext=(cut + 0.9, max(vloss)), fontsize=8, color="grey")
fig.suptitle("Training history, Trial 3 — 5 epochs at 1e-3 then 10 epochs at 1e-5",
             fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "training_curves.png"), dpi=FIG_DPI)
plt.show()

print("")
print("Written to", FIG_DIR)
for f in sorted(os.listdir(FIG_DIR)):
    print("  {:<32} {:>9,} bytes".format(f, os.path.getsize(os.path.join(FIG_DIR, f))))

## Step 11 — Grad-CAM: check what the model is looking at

Accuracy alone cannot tell you *why* a prediction is right. Grad-CAM produces a heatmap of
which pixels drove the decision.

### What you want to see

Heat concentrated on lesions, spots, and discoloured tissue.

### What would be a problem

Heat on the background, on leaf edges, or spread evenly across the frame. That means the
model found a shortcut instead of learning pathology.

A specific shortcut to watch for here: your leaf images come from Bangladesh
(MangoLeafBD) and your fruit images come from India (MangoDHDS). If those two sources
differ in lighting, background, or camera, the model can separate leaf classes from fruit
classes on photographic style alone and never look at a single lesion. It would still score
well on your test set — and fail completely on a Davao farmer's phone.

In [ ]:
def grad_cam(img_batch, class_index=None):
    # Call the real layers directly instead of building a second Model. In Keras 3,
    # base_model.output belongs to the base model's own graph, not to model.inputs,
    # so tf.keras.Model(model.inputs, [base_model.output, ...]) can fail with a
    # graph-disconnected error. Watching the feature map on the tape avoids that.
    with tf.GradientTape() as tape:
        feat = base_model(img_batch, training=False)     # (1, 7, 7, 960)
        tape.watch(feat)
        h = model.get_layer("gap")(feat)
        h = model.get_layer("dropout")(h, training=False)
        logit_out = model.get_layer("logits")(h)
        idx = tf.argmax(logit_out[0]) if class_index is None else class_index
        score = logit_out[:, idx]
    grads = tape.gradient(score, feat)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heat = tf.reduce_sum(feat[0] * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-9)
    return heat.numpy(), int(idx)


# One test image per class.
picked = {}
for c_idx, c in enumerate(CLASS_NAMES):
    d = os.path.join(OUT_DIR, "test", c)
    files = sorted(os.listdir(d))
    if files:
        picked[c] = os.path.join(d, files[0])

fig, axes = plt.subplots(2, len(picked), figsize=(3.2 * len(picked), 7))
for col, (c, path) in enumerate(picked.items()):
    arr = np.array(Image.open(path).convert("RGB").resize(IMG_SIZE, Image.BILINEAR),
                   dtype=np.float32)
    batch = np.expand_dims(arr, 0)
    heat, pred_idx = grad_cam(batch)
    prob = float(model.predict(batch, verbose=0)[0][pred_idx])

    axes[0, col].imshow(arr.astype("uint8"))
    axes[0, col].set_title("actual: {}".format(c), fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(arr.astype("uint8"))
    axes[1, col].imshow(np.array(Image.fromarray(
        (heat * 255).astype("uint8")).resize(IMG_SIZE, Image.BILINEAR)),
        cmap="jet", alpha=0.45)
    ok = "OK" if CLASS_NAMES[pred_idx] == c else "WRONG"
    axes[1, col].set_title("{} {} {:.2f}".format(ok, CLASS_NAMES[pred_idx], prob), fontsize=9)
    axes[1, col].axis("off")

plt.suptitle("Grad-CAM — heat should sit on lesions, not on background", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "grad_cam.png"), dpi=FIG_DPI)
plt.show()

## Step 12 — Temperature scaling

**This is the cell that makes your 70% threshold actually work.**

### The problem in plain terms

Cross-entropy loss rewards a model for being confident, not for being *correctly*
confident. Pushed to 98% accuracy, the model learns to output 0.99 for everything —
including a shoe, a hand, or a banana leaf.

Your app already enforces the manuscript's 70% rule. But nothing ever falls below 0.70, so
the rule filters nothing. The gate is there; nothing ever trips it.

### The fix

Divide all logits by one number T, fitted on the validation set to minimise negative
log-likelihood. T > 1 spreads confidence out; T < 1 sharpens it.

### Why this changes no reported metric

Dividing every logit by the same positive number does not change which one is largest. So
the predicted class is identical for every single image, and accuracy, precision, recall,
F1-score, and the confusion matrix are all **bit-for-bit unchanged.** Only the confidence
number moves.

### Why the app needs no code change

T is folded straight into the final Dense layer's weights and bias:

```
W_new = W / T        b_new = b / T
```

`softmax((Wx + b) / T)` is exactly `softmax(W_new x + b_new)`. The exported model already
produces calibrated confidence. `src/lib/classifier.ts` needs no edit for this.

In [ ]:
from scipy.optimize import minimize_scalar

logit_model = tf.keras.Model(model.inputs, model.get_layer("logits").output)

val_logits = logit_model.predict(val_ds, verbose=0)
val_labels = np.concatenate([y.numpy() for _, y in val_ds])

def nll(T):
    z = val_logits / T
    z = z - z.max(axis=1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return float(-logp[np.arange(len(val_labels)), val_labels].mean())

res = minimize_scalar(nll, bounds=(0.25, 20.0), method="bounded")
T = float(res.x)

print("Validation NLL before: {:.4f}".format(nll(1.0)))
print("Validation NLL after : {:.4f}".format(nll(T)))
print("Fitted temperature T = {:.4f}".format(T))
print("")
if T > 1.05:
    print("T above 1.0 means the model was OVERCONFIDENT. Confidence scores will drop.")
elif T < 0.95:
    print("T below 1.0 means the model was underconfident. Confidence scores will rise.")
else:
    print("T near 1.0 means the model was already reasonably calibrated.")

# Mean confidence before and after, for the record.
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

print("")
print("Mean top-1 confidence on validation:")
print("  before: {:.4f}".format(softmax(val_logits).max(axis=1).mean()))
print("  after : {:.4f}".format(softmax(val_logits / T).max(axis=1).mean()))

In [ ]:
# Fold T into the final Dense layer. No inference-time code change needed anywhere.
acc_before = accuracy_score(y_true, model.predict(test_ds, verbose=0).argmax(axis=1))

W, b = model.get_layer("logits").get_weights()
model.get_layer("logits").set_weights([W / T, b / T])

y_prob_cal = model.predict(test_ds, verbose=0)
acc_after  = accuracy_score(y_true, y_prob_cal.argmax(axis=1))

print("Test accuracy before folding: {:.6f}".format(acc_before))
print("Test accuracy after folding : {:.6f}".format(acc_after))
assert abs(acc_before - acc_after) < 1e-9, "temperature folding changed the predictions"
print("")
print("Identical, as expected. Argmax is invariant to division by a positive constant.")
print("Every metric in the classification report above still stands.")
print("")
print("Mean top-1 confidence on test: {:.4f} -> {:.4f}".format(
    y_prob.max(axis=1).mean(), y_prob_cal.max(axis=1).mean()))

## Step 13 — Rejection thresholds

This measures the exact behaviour you reported: non-mango images receiving a disease label.

### The three signals

| Signal | What it catches |
| :-- | :-- |
| **confidence** — highest probability | model is unsure overall |
| **margin** — top probability minus second | two classes nearly tied, e.g. 0.71 against 0.28 |
| **entropy** — how evenly probability spreads | probability smeared thinly across several classes |

Confidence alone misses the last two. A prediction of 0.71 / 0.28 / 0.01 clears the 70%
bar while being a coin flip between two diseases.

### What gets tuned, and what does not

**Confidence stays fixed at 0.70.** It is a manuscript claim, cited to [31], and this
notebook never touches it. Only margin and entropy are fitted.

### The data

`mango_ood_test/` — your non-mango photographs. This folder is used **nowhere else in this
notebook.** Not in training, not in validation, not in the test evaluation above. It never
produces a gradient and never enters the confusion matrix. It is opened here, once, after
the model is finished.

If the folder contains `tune/` and `report/` subfolders, thresholds are fitted on `tune/`
and reported on `report/`. Otherwise it is split 50/50 with a fixed seed. Fitting and
reporting on the same images would be grading yourself on the answers you studied — the
same reason your mango data has a separate test split.

If the folder does not exist yet, this cell skips itself and the export still runs.

In [ ]:
def prob_stats(p):
    top1 = p.max(axis=1)
    srt  = np.sort(p, axis=1)
    margin = srt[:, -1] - srt[:, -2]
    ent = -(p * np.log(p + 1e-12)).sum(axis=1) / np.log(p.shape[1])
    return top1, margin, ent

def load_flat(paths):
    out = []
    for p in paths:
        try:
            im = Image.open(p).convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
            out.append(np.array(im, dtype=np.float32))
        except Exception as e:
            print("  skipped", os.path.basename(p), repr(e))
    return np.stack(out) if out else None

OOD_AVAILABLE = os.path.isdir(OOD_DIR)

if not OOD_AVAILABLE:
    print("=" * 64)
    print("SKIPPED — no out-of-scope folder at:")
    print("   ", OOD_DIR)
    print("=" * 64)
    print("")
    print("To run this step, create that folder in Google Drive as a SIBLING of")
    print("mango_dataset_v3 (not inside it) and put roughly 150 non-mango photographs in it:")
    print("")
    print("    MyDrive/")
    print("      mango_dataset_v3/    <- unchanged")
    print("      mango_ood_test/")
    print("        tune/              <- ~75 photos, used to fit the thresholds")
    print("        report/            <- ~75 photos, used only for the final number")
    print("")
    print("Shoot other plant leaves, soil, hands, tools, sky, walls, unrelated fruit,")
    print("and a few mango diseases that are NOT among your five. Field conditions,")
    print("same phone. A flat folder with no subfolders also works; it gets split 50/50.")
    print("")
    print("Export in the next step still runs. Rerun this cell after adding the photos.")
    reject_cfg = None
else:
    tune_d, rep_d = os.path.join(OOD_DIR, "tune"), os.path.join(OOD_DIR, "report")
    if os.path.isdir(tune_d) and os.path.isdir(rep_d):
        tune_p = [os.path.join(tune_d, f) for f in sorted(os.listdir(tune_d))
                  if f.lower().endswith(IMG_EXTS)]
        rep_p  = [os.path.join(rep_d, f) for f in sorted(os.listdir(rep_d))
                  if f.lower().endswith(IMG_EXTS)]
        print("Using the tune/ and report/ subfolders as provided.")
    else:
        allp = [os.path.join(OOD_DIR, f) for f in sorted(os.listdir(OOD_DIR))
                if f.lower().endswith(IMG_EXTS)]
        rng = np.random.RandomState(SEED)
        order = rng.permutation(len(allp))
        half = len(allp) // 2
        tune_p = [allp[i] for i in sorted(order[:half])]
        rep_p  = [allp[i] for i in sorted(order[half:])]
        print("Flat folder found. Split 50/50 with seed {}.".format(SEED))

    print("Out-of-scope images  tune: {}   report: {}".format(len(tune_p), len(rep_p)))
    if len(tune_p) < 20 or len(rep_p) < 20:
        print("WARNING: fewer than 20 images on a side. Thresholds will be unstable.")

In [ ]:
if OOD_AVAILABLE:
    X_tune = load_flat(tune_p)
    X_rep  = load_flat(rep_p)

    p_tune = model.predict(X_tune, verbose=0)
    p_rep  = model.predict(X_rep,  verbose=0)

    id_top1,  id_marg,  id_ent  = prob_stats(y_prob_cal)   # in-distribution: mango test set
    ot_top1,  ot_marg,  ot_ent  = prob_stats(p_tune)
    or_top1,  or_marg,  or_ent  = prob_stats(p_rep)

    print("=" * 64)
    print("SIGNAL SEPARATION")
    print("=" * 64)
    print("{:<28} {:>12} {:>12}".format("", "mango test", "non-mango"))
    print("{:<28} {:>12.4f} {:>12.4f}".format("mean confidence", id_top1.mean(), ot_top1.mean()))
    print("{:<28} {:>12.4f} {:>12.4f}".format("mean margin",     id_marg.mean(), ot_marg.mean()))
    print("{:<28} {:>12.4f} {:>12.4f}".format("mean entropy",    id_ent.mean(),  ot_ent.mean()))
    print("")
    print("{:<28} {:>12.1f}% {:>11.1f}%".format(
        "already below 0.70 conf",
        100 * (id_top1 < CONFIDENCE_THRESHOLD).mean(),
        100 * (ot_top1 < CONFIDENCE_THRESHOLD).mean()))
    print("")
    print("The wider the gap between the two columns, the better rejection can work.")
    print("If the non-mango column looks like the mango column, confidence alone cannot")
    print("separate them and margin plus entropy are doing the real work.")

    # Distribution plots.
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (idv, oodv, name, line) in zip(axes, [
            (id_top1, ot_top1, "confidence", CONFIDENCE_THRESHOLD),
            (id_marg, ot_marg, "margin",     None),
            (id_ent,  ot_ent,  "entropy",    None)]):
        ax.hist(idv,  bins=30, alpha=0.6, label="mango test", density=True)
        ax.hist(oodv, bins=30, alpha=0.6, label="non-mango",  density=True)
        if line is not None:
            ax.axvline(line, ls="--", c="red", label="0.70 threshold")
        ax.set_title(name); ax.legend(fontsize=8)
    fig.suptitle("Rejection signals: mango test set against non-mango images",
                 fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "rejection_distributions.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
if OOD_AVAILABLE:
    # Fit margin and entropy on the tune half. Confidence stays pinned at 0.70.
    best = None
    for m in np.arange(0.00, 0.96, 0.01):
        for e in np.arange(1.00, 0.04, -0.01):
            keep_id  = (id_top1 >= CONFIDENCE_THRESHOLD) & (id_marg >= m) & (id_ent <= e)
            keep_ood = (ot_top1 >= CONFIDENCE_THRESHOLD) & (ot_marg >= m) & (ot_ent <= e)
            retention = float(keep_id.mean())
            rejection = float(1.0 - keep_ood.mean())
            if retention >= MIN_ID_RETENTION:
                if best is None or rejection > best["rejection"]:
                    best = {"margin": float(m), "entropy": float(e),
                            "retention": retention, "rejection": rejection}

    if best is None:
        print("No threshold pair reached {:.0f}% in-distribution retention.".format(
            100 * MIN_ID_RETENTION))
        print("Falling back to confidence only.")
        best = {"margin": 0.0, "entropy": 1.0,
                "retention": float((id_top1 >= CONFIDENCE_THRESHOLD).mean()),
                "rejection": float(1.0 - (ot_top1 >= CONFIDENCE_THRESHOLD).mean())}

    MARGIN_THRESHOLD  = best["margin"]
    ENTROPY_THRESHOLD = best["entropy"]

    # Honest number: evaluate on the report half, never used for fitting.
    keep_rep = ((or_top1 >= CONFIDENCE_THRESHOLD) &
                (or_marg >= MARGIN_THRESHOLD) &
                (or_ent  <= ENTROPY_THRESHOLD))
    rejection_report = float(1.0 - keep_rep.mean())

    keep_id_final = ((id_top1 >= CONFIDENCE_THRESHOLD) &
                     (id_marg >= MARGIN_THRESHOLD) &
                     (id_ent  <= ENTROPY_THRESHOLD))
    acc_on_kept = float(accuracy_score(y_true[keep_id_final],
                                       y_prob_cal.argmax(axis=1)[keep_id_final]))

    print("=" * 64)
    print("RECOMMENDED THRESHOLDS")
    print("=" * 64)
    print("  confidence >= {:.2f}     (FIXED by manuscript, not tuned)".format(
        CONFIDENCE_THRESHOLD))
    print("  margin     >= {:.2f}".format(MARGIN_THRESHOLD))
    print("  entropy    <= {:.2f}".format(ENTROPY_THRESHOLD))
    print("")
    print("Measured on the tune half:")
    print("  in-distribution retention : {:.1f}%".format(100 * best["retention"]))
    print("  out-of-scope rejection    : {:.1f}%".format(100 * best["rejection"]))
    print("")
    print("Measured on the held-out report half (THIS is the manuscript number):")
    print("  out-of-scope rejection    : {:.1f}%".format(100 * rejection_report))
    print("")
    print("Accuracy on the mango predictions that pass all three checks: {:.2f}%".format(
        100 * acc_on_kept))
    print("")
    print("Read this as: of every 100 non-mango photographs, about {:.0f} are correctly".format(
        100 * rejection_report))
    print("flagged Uncertain, while {:.0f} of every 100 real mango scans still get a".format(
        100 * best["retention"]))
    print("diagnosis. The remaining out-of-scope images still slip through. That is the")
    print("closed-set limitation to state in Scope and Delimitation.")

    reject_cfg = {
        "confidenceThreshold": CONFIDENCE_THRESHOLD,
        "marginThreshold": round(MARGIN_THRESHOLD, 4),
        "entropyThreshold": round(ENTROPY_THRESHOLD, 4),
        "tuneRetention": round(best["retention"], 4),
        "tuneRejection": round(best["rejection"], 4),
        "reportRejection": round(rejection_report, 4),
        "accuracyOnAccepted": round(acc_on_kept, 4),
        "oodTuneCount": len(tune_p),
        "oodReportCount": len(rep_p),
    }
else:
    MARGIN_THRESHOLD, ENTROPY_THRESHOLD = 0.0, 1.0
    reject_cfg = None
    print("Rejection thresholds not fitted. Only the 70% confidence check will apply.")

## Step 14 — Export to TensorFlow.js

Two steps, and the order matters.

`model.export()` writes a TensorFlow SavedModel directory. The `.h5` route fails in the
browser with:

```
Error: An InputLayer should be passed either a batchInputShape or an inputShape.
```

That is the Keras 3 serialization bug from Trial 1. The converter then takes the SavedModel
with `--input_format=tf_saved_model`, and the browser loads the result with
`tf.loadGraphModel`.

The exported graph contains the trained network only. The augmentation layers were applied
in the `tf.data` pipeline, not stacked inside the model, so no random operation is baked
into the graph — the same photograph produces the same answer every time.

The temperature is already folded into the weights, so what ships is calibrated.

### The converter repairs itself before running

On Colab's Python 3.13 the conversion chain hits three unrelated packaging problems, so the
cell diagnoses and repairs each one instead of failing. It prints which repairs it applied.

1. **`AttributeError: module 'numpy' has no attribute 'object'`.** pip cannot install
   `tensorflowjs` 4.x here because that release requires `tensorflow-decision-forests`, which
   has no 3.13 wheel, so pip silently falls back to an ancient 3.x build that still uses the
   numpy aliases removed in numpy 1.24. The cell rewrites those aliases inside the installed
   package, then upgrades to 4.22.0 with `--no-deps` so pip cannot change the TensorFlow
   version underneath a model that is already trained and sitting in memory.
2. **`module 'tensorflow.compat.v1' has no attribute 'estimator'`.** `tensorflow_hub` 0.12.0
   subclasses the Estimator API at import time, and TensorFlow removed Estimator in 2.16.
   The cell upgrades `tensorflow_hub` to 0.16.1, which dropped that subclass.
3. **`No module named 'distutils'`.** Python 3.13 removed it and pip cannot reinstall it.
   The cell writes a stub into `/content/tfjs_stubs` and passes that directory to the
   converter subprocess through `PYTHONPATH` only. Nothing in `dist-packages` is touched, so
   the running notebook keeps the real packages.

Repairs 1 and 2 are attempted up front. Anything else is handled by a loop that reads the
converter's own traceback, identifies the package that failed to import, tries upgrading it
once, and stubs it if the upgrade does not help. Up to 8 rounds. TensorFlow, numpy, and
tensorflowjs are never stubbed.

If every round fails the cell raises with the converter's output. The trained SavedModel is
already on disk by then, so nothing needs retraining.

In [ ]:
SAVED_MODEL = os.path.join(EXPORT_DIR, "saved_model")
TFJS_DIR    = os.path.join(EXPORT_DIR, "model")

shutil.rmtree(SAVED_MODEL, ignore_errors=True)
shutil.rmtree(TFJS_DIR, ignore_errors=True)

model.export(SAVED_MODEL)
print("SavedModel written to", SAVED_MODEL)

In [ ]:
import re, os, sys, glob, json, subprocess
import importlib, importlib.util
import importlib.metadata as importlib_metadata

# Stubs live here, not in dist-packages, and are handed to the converter through
# PYTHONPATH. Real installed packages are left untouched.
STUB_DIR = "/content/tfjs_stubs"

# Module name as imported -> name pip knows it by. None means no wheel exists for
# this Python version, so the module gets stubbed instead of installed.
PIP_NAME = {
    "tensorflow_decision_forests": None,
    "tensorflow_hub": "tensorflow-hub",
    "tf_keras": "tf-keras",
    "jax": "jax",
    "jaxlib": "jaxlib",
    "flax": "flax",
    "packaging": "packaging",
}

# Never stub these. If the failure is really inside TensorFlow itself, stubbing it
# would hide the problem rather than fix it.
NEVER_STUB = ("tensorflow", "numpy", "tensorflowjs")


def tfjs_dirs():
    # find_spec locates the package without running its __init__, so this works
    # even when importing tensorflowjs would raise.
    importlib.invalidate_caches()
    spec = importlib.util.find_spec("tensorflowjs")
    if spec is None or not spec.submodule_search_locations:
        return None
    return list(spec.submodule_search_locations)[0]


def version_of(dist):
    importlib.invalidate_caches()
    try:
        return importlib_metadata.version(dist)
    except Exception:
        return "not installed"


def patch_numpy_aliases(pkg_dir):
    # numpy 1.24 removed np.object, np.bool, np.int, np.float and np.str. Older
    # tensorflowjs builds reference them at import time. The \b boundary keeps the
    # rewrite off np.object_, np.bool_, np.float32 and friends. Re-run after every
    # pip install, in case pip moved to an older build.
    if not pkg_dir:
        return []
    names = ("object", "bool", "int", "float", "str", "complex")
    touched = []
    for path in glob.glob(os.path.join(pkg_dir, "**", "*.py"), recursive=True):
        try:
            with open(path, encoding="utf-8") as fh:
                src = fh.read()
        except (OSError, UnicodeDecodeError):
            continue
        new = src
        for n in names:
            new = re.sub(r"\bnp\.%s\b" % n, n, new)
            new = re.sub(r"\bnumpy\.%s\b" % n, n, new)
        if new != src:
            with open(path, "w", encoding="utf-8") as fh:
                fh.write(new)
            touched.append(os.path.relpath(path, pkg_dir))
    return touched


def write_stub(module_name):
    # Permissive placeholder. tensorflowjs imports these at load time but only uses
    # them for input formats this project never produces, so an empty shell is
    # enough to get past the import.
    d = os.path.join(STUB_DIR, module_name)
    os.makedirs(d, exist_ok=True)
    body = (
        "__version__ = '0.0.0-stub'\n"
        "\n"
        "class _Any(object):\n"
        "    def __getattr__(self, k):\n"
        "        return _Any\n"
        "    def __call__(self, *a, **k):\n"
        "        return _Any()\n"
        "\n"
        "def __getattr__(name):\n"
        "    return _Any\n"
    )
    for fname in ("__init__.py", "keras.py"):
        with open(os.path.join(d, fname), "w") as fh:
            fh.write(body)


def pip_install(pip_name, upgrade=True):
    # --no-deps is deliberate. A plain install can change the TensorFlow version,
    # which forces a runtime restart and loses the trained model held in memory.
    args = [sys.executable, "-m", "pip", "install", "-q", "--no-deps"]
    if upgrade:
        args.append("-U")
    args.append(pip_name)
    return subprocess.run(args, capture_output=True, text=True).returncode == 0


def run_converter():
    # List form, no shell. A shell would glob-expand the * in --quantize_float16=*.
    quant = ["--quantize_float16=*"] if QUANTIZE_FLOAT16 else []
    cmd = (["tensorflowjs_converter", "--input_format=tf_saved_model"]
           + quant + [SAVED_MODEL, TFJS_DIR])
    env = dict(os.environ)
    env["TF_CPP_MIN_LOG_LEVEL"] = "2"
    if os.path.isdir(STUB_DIR):
        env["PYTHONPATH"] = STUB_DIR + os.pathsep + env.get("PYTHONPATH", "")
    print("     $", " ".join(cmd))
    return subprocess.run(cmd, capture_output=True, text=True, env=env)


def blame(text):
    """Which third-party package is failing to import inside the converter."""
    text = text or ""
    m = re.search(r"No module named '([^']+)'", text)
    if m:
        return m.group(1).split(".")[0]
    # Otherwise take the deepest traceback frame that sits in site-packages and
    # belongs to neither tensorflowjs nor tensorflow.
    found = None
    for path in re.findall(r'File "([^"]+)", line \d+, in <module>', text):
        parts = path.replace("\\", "/").split("/")
        for marker in ("dist-packages", "site-packages"):
            if marker in parts:
                top = parts[parts.index(marker) + 1]
                if top.endswith(".py"):
                    top = top[:-3]
                if top not in NEVER_STUB:
                    found = top
    return found


def tail(text, n=25):
    lines = [l for l in (text or "").strip().split("\n") if l.strip()]
    return "\n".join(lines[-n:])


os.makedirs(STUB_DIR, exist_ok=True)
pkg = tfjs_dirs()
print("numpy           :", np.__version__)
print("tensorflow      :", tf.__version__)
print("tensorflowjs    :", version_of("tensorflowjs"))
print("tensorflow_hub  :", version_of("tensorflow-hub"))
print("package dir     :", pkg)
print("")
if pkg is None:
    raise RuntimeError("tensorflowjs is not installed. Run cell #1, then this cell again.")

# --- Step A: tensorflow_hub 0.15 and older import tf.compat.v1.estimator, -----
# --- which TensorFlow removed in 2.16. Move to a release that does not. --------
print("Step A - upgrading tensorflow_hub (old builds need the removed Estimator API)")
pip_install("tensorflow-hub")
print("        tensorflow_hub :", version_of("tensorflow-hub"))
patch_numpy_aliases(tfjs_dirs())
proc = run_converter()

# --- Step B: repair loop. Upgrade the offending package once, then stub it. ----
tried = {}
for _ in range(8):
    if proc.returncode == 0:
        break
    mod = blame((proc.stderr or "") + "\n" + (proc.stdout or ""))
    if mod is None or mod in NEVER_STUB:
        break
    pip_name = PIP_NAME.get(mod, mod)
    stage = tried.get(mod, 0)
    print("")
    if stage == 0 and pip_name:
        print("Step B - {} is breaking the import, upgrading it".format(mod))
        if not pip_install(pip_name):
            print("        pip could not install {}".format(pip_name))
        else:
            print("        {} :".format(mod), version_of(pip_name))
        tried[mod] = 1
    elif stage <= 1:
        print("Step B - {} still unusable, shadowing it with a stub".format(mod))
        print("        the SavedModel path does not use it")
        write_stub(mod)
        tried[mod] = 2
    else:
        break
    patch_numpy_aliases(tfjs_dirs())
    proc = run_converter()

# --- Report -------------------------------------------------------------------
model_json = os.path.join(TFJS_DIR, "model.json")
if proc.returncode != 0 or not os.path.isfile(model_json):
    print("")
    print("=" * 70)
    print("CONVERSION FAILED - exit code", proc.returncode)
    print("=" * 70)
    print(tail(proc.stderr) or tail(proc.stdout))
    print("")
    print("Nothing is lost. The trained, temperature-folded model is on disk at:")
    print("   ", SAVED_MODEL)
    print("Send the lines above and I will send a replacement for this cell.")
    raise RuntimeError("tensorflowjs_converter failed, see output above")

stubbed = sorted(d for d in os.listdir(STUB_DIR)
                 if os.path.isdir(os.path.join(STUB_DIR, d)))
print("")
print("Conversion succeeded with tensorflowjs", version_of("tensorflowjs"))
if stubbed:
    print("Stubbed for the converter only:", ", ".join(stubbed))
warn = tail(proc.stderr, 5)
if warn:
    print("")
    print("Converter notes (warnings here are normal):")
    print(warn)

# What the browser will load. These names must match src/lib/classifier.ts.
with open(model_json) as fh:
    mj = json.load(fh)
sig = mj.get("signature", {})
print("")
print("format       :", mj.get("format"), "| generated by tfjs", mj.get("generatedBy"))
print("inputs       :", list(sig.get("inputs", {}).keys()))
print("outputs      :", list(sig.get("outputs", {}).keys()))
print("weight shards:", len(mj.get("weightsManifest", [{}])[0].get("paths", [])))

print("")
total = 0
for f in sorted(os.listdir(TFJS_DIR)):
    p = os.path.join(TFJS_DIR, f)
    if not os.path.isfile(p):
        continue
    size = os.path.getsize(p)
    total += size
    print("  {:<28} {:>10,} bytes  ({:.2f} MB)".format(f, size, size / 1048576))
print("  {:<28} {:>10,} bytes  ({:.2f} MB)".format("TOTAL", total, total / 1048576))
print("")
print("Trial 2 shipped 11.51 MB across 3 shards, unquantized.")
if not QUANTIZE_FLOAT16:
    print("")
    print("NOTE: QUANTIZE_FLOAT16 is False, so these weights are float32.")
    print("The comment at src/lib/classifier.ts:37 claims the model is Float16 quantized.")
    print("Either set QUANTIZE_FLOAT16 = True and rerun, or correct that comment.")

## Step 15 — Config for the app, and the run record

Writes `model_config.json` next to the model. It carries the class order, the fitted
temperature, and the three thresholds, so nothing has to be retyped from a screenshot.

Also writes `run_record.json` with the numbers `docs/MODEL_VERSIONING.md` asks every
completed run to record.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

model_config = {
    "modelVersion": "model-v3.0.0",
    "trial": "3",
    "classes": CLASS_NAMES,
    "inputSize": list(IMG_SIZE),
    "pixelRange": "0-255 raw, MobileNetV3 rescales internally",
    "temperature": round(T, 6),
    "temperatureFoldedIntoWeights": True,
    "thresholds": reject_cfg if reject_cfg else {
        "confidenceThreshold": CONFIDENCE_THRESHOLD,
        "marginThreshold": None,
        "entropyThreshold": None,
        "note": "out-of-scope set not available; only the 70% confidence check is calibrated",
    },
}

run_record = {
    "trial": "3",
    "notebook": "MangoLeaf_AI_Training_v3.ipynb",
    "datasetSource": RAW_DIR,
    "splitSeed": SEED,
    "duplicateScan": {
        "enabled": bool(REMOVE_DUPLICATES),
        "phashDistance": PHASH_DISTANCE,
        "imagesScanned": len(kept),
        "duplicateGroups": len(dup_groups),
        "redundantRemoved": int(len(kept) - len(pool)),
        "crossLabelPairs": len(cross),
        "nearDuplicateLeakagePairs": len(straddle),
        "corruptFiles": len(corrupt),
    },
    "splitCounts": {
        c: {s: len(os.listdir(os.path.join(OUT_DIR, s, c)))
            for s in ("train", "val", "test")} for c in CLASS_NAMES
    },
    "classWeights": {CLASS_NAMES[i]: round(w, 4) for i, w in class_weight.items()},
    "augmentation": {
        "appliedAt": "tf.data pipeline, live, every epoch",
        "rotationDegrees": AUG_ROTATION_DEG,
        "flip": AUG_FLIP,
        "zoom": AUG_ZOOM,
        "brightness": AUG_BRIGHTNESS,
        "shear": AUG_SHEAR,
    },
    "architecture": "MobileNetV3-Large + GAP + Dropout(0.2) + Dense(6) + Softmax",
    "stages": [
        {"stage": 1, "epochs": STAGE1_EPOCHS, "lr": STAGE1_LR, "baseFrozen": True},
        {"stage": 2, "epochs": STAGE2_EPOCHS, "lr": STAGE2_LR,
         "baseFrozen": False, "batchNormFrozen": True},
    ],
    "metrics": {
        "testAccuracy": round(float(test_acc), 4),
        "testLoss": round(float(test_loss), 4),
        "bestValLoss": round(float(min(vloss)), 4),
        "bestValEpoch": int(np.argmin(vloss)) + 1,
    },
    "perClass": classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                      digits=4, output_dict=True),
    "confusionMatrix": cm.tolist(),
    "calibration": {"temperature": round(T, 6),
                    "valNllBefore": round(nll(1.0), 6),
                    "valNllAfter": round(nll(T), 6)},
    "rejection": reject_cfg,
    "export": {
        "format": "TensorFlow.js Graph Model via tf_saved_model",
        "quantizeFloat16": bool(QUANTIZE_FLOAT16),
        "files": {f: {"bytes": os.path.getsize(os.path.join(TFJS_DIR, f)),
                      "sha256": sha256(os.path.join(TFJS_DIR, f))}
                  for f in sorted(os.listdir(TFJS_DIR))},
        "totalBytes": sum(os.path.getsize(os.path.join(TFJS_DIR, f))
                          for f in os.listdir(TFJS_DIR)),
    },
}

with open(os.path.join(TFJS_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
with open(os.path.join(EXPORT_DIR, "run_record.json"), "w") as f:
    json.dump(run_record, f, indent=2)

print(json.dumps(model_config, indent=2))

In [ ]:
os.makedirs(DRIVE_SAVE, exist_ok=True)
zip_path = os.path.join(DRIVE_SAVE, "mangoleaf_model_v3.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(TFJS_DIR)):
        z.write(os.path.join(TFJS_DIR, f), f)
    z.write(os.path.join(EXPORT_DIR, "run_record.json"), "run_record.json")
    for f in sorted(os.listdir(FIG_DIR)):
        z.write(os.path.join(FIG_DIR, f), os.path.join("figures", f))

# Also drop the figures into Drive unzipped, so they can be inserted into Word
# without extracting anything.
drive_figs = os.path.join(DRIVE_SAVE, "figures")
shutil.rmtree(drive_figs, ignore_errors=True)
shutil.copytree(FIG_DIR, drive_figs)
shutil.copy(os.path.join(EXPORT_DIR, "run_record.json"),
            os.path.join(DRIVE_SAVE, "run_record.json"))

print("Saved to Drive:")
print("  ", zip_path, "({:.2f} MB)".format(os.path.getsize(zip_path) / 1048576))
print("  ", os.path.join(DRIVE_SAVE, "run_record.json"))
print("  ", drive_figs + "/")
for f in sorted(os.listdir(drive_figs)):
    print("       ", f)

## Step 16 — What to do with the output

### 1. Install the model

Unzip and replace the contents of `public/model/` in the app. Then bump `modelVersion` in
`src/lib/constants.ts` to `model-v3.0.0`.

### 2. Verify the class order

`model_config.json` lists `classes`. It must match `CLASSES` in `src/lib/constants.ts`
element for element, in that order. Keras assigns label indices alphabetically; if the two
lists disagree, every prediction is silently mislabelled and accuracy metrics will still
look fine.

### 3. Fix the rounding bug

`src/lib/classifier.ts:127` rounds before comparing:

```ts
const confidence = Math.round(maxProb * 100) / 100;
const diseaseName = confidence >= 0.70 ? CLASSES[maxIndex] : "Uncertain / Unrecognized";
```

A raw probability of `0.6951` becomes `0.70` and passes. The real floor is 69.5%, not the
70% the manuscript claims in four places. Compare the raw value; round only for display.

### 4. Add the margin and entropy checks

`model_config.json` carries `marginThreshold` and `entropyThreshold`. Both are computed
from the same probability array the app already has — no second inference pass, no extra
model output.

```
margin  = highest - second_highest
entropy = -sum(p * ln(p)) / ln(6)
```

Accept only when all three hold: confidence >= 0.70, margin >= marginThreshold, and
entropy <= entropyThreshold.

### 5. Insert the figures

`MyDrive/mangoleaf_v3/figures/` holds every figure as a 200 DPI PNG, ready to insert
into Word directly. `per_class_metrics.csv` opens in Excel or Google Sheets if you would
rather build the classification-report table by hand than paste the printed one.

Give each figure a caption and a number in the order they appear in your text, and refer
to every one of them at least once in the body. A figure no sentence points at reads as
padding to a panel.

### 6. Fill in the manuscript

`docs/MANUSCRIPT_MODIFICATIONS.md` has a table titled "Items pending measured results".
`run_record.json` supplies every value in it.

### 7. Write the trial report

Create `docs/model-training/TRIAL_3.md` from `run_record.json`, following the shape of
`TRIAL_2.md`.

While there: `TRIAL_2.md:48` says only `bacterial_black_spot` and `mango_scab` were
pre-augmented, but the notebook's `fruit_targets` list also contained `healthy`.
`TRIAL_2.md:50` claims 46,500 real-time augmented exposures in RAM, and Trial 2 had no
training-time augmentation at all. Both should be corrected so Trial 2 stays a trustworthy
comparison baseline.

---

## Honest limits of this run

**Rejection is improved, not solved.** A six-class softmax always distributes probability
across exactly those six classes. There is no output node meaning "none of the above", so
some out-of-scope images will still land on a disease with genuine-looking confidence. The
`reportRejection` figure is the measured rate — quote it as a limitation rather than
implying the problem is gone.

**Out-of-scope diseases are the hardest case.** A mango disease that is not one of your
five still looks like mango tissue with lesions. Every feature the model relies on is
present. These will be rejected less often than a photograph of a shoe.

**Compare against 97.12%, not 98.25%.** Trial 2's 98.25% was partly measured on images the
model had already memorised, so it is not a baseline. Trial 2.1's 97.12% on 417 deduplicated
test images is. A Trial 3 figure a point either side of it is the same model performance
measured on a different 393 images.

**Grad-CAM is diagnostic, not proof.** Heat sitting on lesions is reassuring. It does not
guarantee the model generalises to a Davao orchard, because the training photographs still
come mostly from Bangladesh and India.

---

## If something looks wrong

| Symptom | Likely cause |
| :-- | :-- |
| Test accuracy above 97% with several classes at 1.00 F1 | Duplicates remain. Raise `PHASH_DISTANCE` to 8 and rerun from the scan. |
| Test accuracy below 80% | Check the augmentation preview grid. Too-aggressive shear or brightness can destroy lesion detail. |
| Stage 2 epoch 1 drops sharply again | The BatchNormalization freeze did not apply. Confirm the frozen count printed non-zero and that `model.compile` ran after the freeze. |
| `T` fitted near 20 (the bound) | Severe overconfidence, usually from heavy train/test leakage. Fix the duplicates first. |
| Rejection below 40% | Expected without a negative class, especially if the out-of-scope photos resemble foliage. More visually varied non-mango images help. |
| All 8 augmentation panels identical | `augment` is not being called with `training=True`. |

In [ ]:
# How many test images have a same-label near neighbour in train, by threshold?
# Deduplication at PHASH_DISTANCE guarantees within-folder survivors are further
# apart than the threshold, so anything appearing here at 6 and above is a
# same-leaf burst that dedup deliberately left alone. If the four Roboflow
# folders light up and the two clean folders do not, their perfect scores are
# leakage rather than skill.
try:
    manifest, D, chosen, kept
except NameError:
    raise RuntimeError("Session state is gone. Rerun cells #4 through #7 first.")

info = {}
for split, entries in manifest.items():
    for src, dst, tgt in entries:
        info[src] = (split, tgt)

paths = [kept[i][0] for i in chosen]
by_split = {"train": [], "val": [], "test": []}
for k, p in enumerate(paths):
    s = info.get(p, (None, None))[0]
    if s:
        by_split[s].append(k)

sub = D[np.ix_(chosen, chosen)]
tr = np.array(by_split["train"])
te = np.array(by_split["test"])
block = sub[np.ix_(te, tr)]

THRESHOLDS = (5, 6, 8, 10, 12, 15)
hdr = "{:<24}{:>8}" + "{:>9}" * len(THRESHOLDS)
print(hdr.format("class", "support", *["d<={}".format(t) for t in THRESHOLDS]))
print("-" * (32 + 9 * len(THRESHOLDS)))

for c in CLASS_NAMES:
    te_is_c = np.array([info[paths[i]][1] == c for i in te])
    tr_is_c = np.array([info[paths[j]][1] == c for j in tr])
    cells = []
    for t in THRESHOLDS:
        hit = (block[:, tr_is_c] <= t).any(axis=1)
        cells.append(int((hit & te_is_c).sum()))
    print(hdr.format(c, int(te_is_c.sum()), *cells))

tot = []
for t in THRESHOLDS:
    hit = np.zeros(len(te), dtype=bool)
    for c in CLASS_NAMES:
        te_is_c = np.array([info[paths[i]][1] == c for i in te])
        tr_is_c = np.array([info[paths[j]][1] == c for j in tr])
        hit |= (block[:, tr_is_c] <= t).any(axis=1) & te_is_c
    tot.append(int(hit.sum()))
print("-" * (32 + 9 * len(THRESHOLDS)))
print(hdr.format("TOTAL", len(te), *tot))
print("")
print("Zero in the d<=5 column is expected and confirms deduplication worked.")
print("Read the d<=8 and d<=10 columns against the per-class F1 scores:")
try:
    from sklearn.metrics import f1_score
    f1s = f1_score(y_true, y_pred, average=None,
                   labels=list(range(len(CLASS_NAMES))))
    for i in np.argsort(-f1s):
        print("  {:<22} {:.4f}".format(CLASS_NAMES[i], f1s[i]))
except NameError:
    print("  (run cell #16 first and rerun this cell to list the F1 scores)")
print("Counts that track the F1 ranking point to same-leaf leakage, and the fix")
print("for v3 is to split by physical leaf group rather than by individual file.")

In [ ]:
# Measures what float16 weight storage costs, using the real test set.
# Rounding a float32 weight to float16 and back reproduces exactly what tfjs
# holds after it unpacks a quantized model, so this is not an approximation of
# the browser: it is the same arithmetic.
from sklearn.metrics import accuracy_score

orig = [w.copy() for w in model.get_weights()]
n_w = sum(w.size for w in orig)

# float16 range check. Anything above 65504 becomes inf, and anything below
# 6.1e-5 loses precision, below 5.96e-8 flushes to zero. Overflow would be a
# hard stop; underflow in a few tiny weights is harmless.
big = sum(int((np.abs(w) > 65504).sum()) for w in orig)
sub = sum(int(((np.abs(w) > 0) & (np.abs(w) < 6.104e-5)).sum()) for w in orig)
zero = sum(int(((np.abs(w) > 0) & (np.abs(w) < 5.96e-8)).sum()) for w in orig)

print("Weights                     : {:,}".format(n_w))
print("Above float16 max (overflow): {:,}".format(big))
print("Subnormal in float16        : {:,}  ({:.3f}%)".format(sub, 100 * sub / n_w))
print("Would flush to zero         : {:,}  ({:.3f}%)".format(zero, 100 * zero / n_w))
print("")

p32 = model.predict(test_ds, verbose=0)
acc32 = accuracy_score(y_true, p32.argmax(axis=1))

model.set_weights([w.astype(np.float16).astype(np.float32) for w in orig])
p16 = model.predict(test_ds, verbose=0)
acc16 = accuracy_score(y_true, p16.argmax(axis=1))

model.set_weights(orig)  # restore immediately, before anything else runs
assert np.allclose(model.predict(test_ds, verbose=0), p32, atol=0), \
    "restore failed, rerun cells #13 through #22"

flipped = int((p32.argmax(axis=1) != p16.argmax(axis=1)).sum())
c32, c16 = p32.max(axis=1), p16.max(axis=1)
gate = int(((c32 >= CONFIDENCE_THRESHOLD) != (c16 >= CONFIDENCE_THRESHOLD)).sum())

print("Test accuracy, float32 weights : {:.6f}".format(acc32))
print("Test accuracy, float16 weights : {:.6f}".format(acc16))
print("Difference                     : {:+.6f}  ({:+d} images of {})".format(
    acc16 - acc32, int(round((acc16 - acc32) * len(y_true))), len(y_true)))
print("")
print("Predicted class changed        : {} of {} images".format(flipped, len(y_true)))
print("Crossed the {:.0%} gate differently: {} images".format(CONFIDENCE_THRESHOLD, gate))
print("Largest top-1 confidence shift : {:.6f}".format(np.abs(c32 - c16).max()))
print("Mean top-1 confidence shift    : {:.6f}".format(np.abs(c32 - c16).mean()))
print("")
print("Original weights restored. The model is exactly as it was.")
print("")
print("Decision rule:")
print("  overflow 0, accuracy difference 0.000000, gate crossings 0")
print("     -> float16 is free. Take the 5.7 MB.")
print("  overflow 0, one or two images move")
print("     -> still take it. Your WEBGL_FORCE_F16_TEXTURES setting already")
print("        truncates weights to float16 on the GPU, so the browser is")
print("        paying this cost today while downloading float32 anyway.")
print("  overflow above 0")
print("     -> do NOT quantize. Keep float32 and fix the comment instead.")


In [ ]:
# Reads the converted model back off disk and checks what actually shipped.
import json

mj = json.load(open(os.path.join(TFJS_DIR, "model.json")))
manifest = mj["weightsManifest"]

buf = b""
for group in manifest:
    for p in group["paths"]:
        with open(os.path.join(TFJS_DIR, p), "rb") as fh:
            buf += fh.read()

ITEMSIZE = {"float32": 4, "float16": 2, "int32": 4, "uint8": 1, "uint16": 2, "bool": 1}
NPTYPE = {"float32": np.float32, "float16": np.float16, "int32": np.int32,
          "uint8": np.uint8, "uint16": np.uint16, "bool": np.bool_}

off = 0
quantized = 0
left_float32 = []
nonfinite = []
n_values = 0

for group in manifest:
    for w in group["weights"]:
        n = 1
        for s in w.get("shape", []):
            n *= s
        stored = w.get("quantization", {}).get("dtype", w["dtype"])
        nbytes = n * ITEMSIZE[stored]
        raw = buf[off:off + nbytes]
        off += nbytes
        n_values += n
        if "quantization" in w:
            quantized += 1
        elif w["dtype"] == "float32":
            left_float32.append((w["name"], n))
        if stored.startswith("float"):
            arr = np.frombuffer(raw, dtype=NPTYPE[stored]).astype(np.float32)
            bad = int((~np.isfinite(arr)).sum())
            if bad:
                nonfinite.append((w["name"], bad, n))

print("Bytes in shards        : {:,}".format(len(buf)))
print("Bytes walked           : {:,}".format(off))
assert off == len(buf), "manifest and shard bytes disagree, conversion is corrupt"
print("Weight tensors         :", quantized + len(left_float32))
print("  quantized to float16 :", quantized)
print("  left as float32      :", len(left_float32))
print("Scalar values          : {:,}".format(n_values))
print("")

if left_float32:
    print("Tensors the converter refused to quantize (out of float16 range):")
    for name, n in left_float32[:10]:
        print("   {:<60} {:>10,}".format(name, n))
    print("This is the converter protecting itself. Nothing to fix.")
    print("")

if nonfinite:
    print("inf or NaN found in shipped weights:")
    for name, bad, n in nonfinite[:10]:
        print("   {:<60} {:>6} of {:,}".format(name, bad, n))
    print("")
    print("Cell #31 measured this exact condition and accuracy did not move,")
    print("so this is expected rather than a fault. Note it in the trial report.")
else:
    print("No inf or NaN anywhere in the shipped weights.")

print("")
total = sum(os.path.getsize(os.path.join(TFJS_DIR, f))
            for f in os.listdir(TFJS_DIR)
            if os.path.isfile(os.path.join(TFJS_DIR, f)))
print("Total shipped size     : {:,} bytes  ({:.2f} MB)".format(total, total / 1048576))
print("Unquantized was        : 12,094,260 bytes  (11.53 MB)")
print("Saved                  : {:,} bytes  ({:.1f}%)".format(
    12094260 - total, 100 * (12094260 - total) / 12094260))
print("")
print("Under 6 MB means the quantization took effect. Around 11.5 MB means the")
print("flag did not reach the converter, so cell #3 was not rerun before #27.")


## Step 17 — Preserve the trained weights for later threshold fitting

**Run this cell. It is not optional.**

Step 16 saves the web model to Drive. It does **not** save the Keras model, which lives
only on Colab's local disk and is deleted when the session ends.

The margin and entropy rejection thresholds are fitted by cells that call `model.predict()`
on out-of-scope photographs. If those photographs are not ready yet — the usual case, since
collecting roughly 150 of them takes a separate trip — then the thresholds must be fitted in
a **later session**, after this one is gone.

Without this cell, that later session has no model to fit against, and the only way forward
is a full retrain. A retrained model has different weights than the one already shipped to
the app, so thresholds fitted on it would not match what users are running.

This cell writes the calibrated Keras model plus the test-set probabilities to
`MyDrive/mangoleaf_v3/resume/`. Roughly 35 MB, a few seconds. The file is larger than the
5.88 MB web model because the `.keras` format also stores the Adam optimizer state, which is
two extra slots per weight.


In [ ]:
# Save everything needed to fit the rejection thresholds LATER, in a new session,
# without retraining. Run this while the current session is still alive.
import numpy as np

RESUME_DIR = os.path.join(DRIVE_SAVE, "resume")
os.makedirs(RESUME_DIR, exist_ok=True)

# 1. The calibrated model, exactly as exported to the web (temperature folded in).
keras_path = os.path.join(RESUME_DIR, "mangoleaf_v3_calibrated.keras")
model.save(keras_path)

# 2. The test-set probabilities and labels, so the in-distribution side of the
#    threshold sweep does not need the dataset rebuilt.
np.save(os.path.join(RESUME_DIR, "y_prob_cal.npy"), y_prob_cal)
np.save(os.path.join(RESUME_DIR, "y_true.npy"), y_true)

# 3. A tiny record of what these files belong to.
with open(os.path.join(RESUME_DIR, "resume_info.json"), "w") as f:
    json.dump({
        "trial": "3",
        "temperature": round(float(T), 6),
        "temperatureFoldedIntoWeights": True,
        "testAccuracy": round(float(test_acc), 6),
        "classes": CLASS_NAMES,
        "confidenceThreshold": CONFIDENCE_THRESHOLD,
        "minIdRetention": MIN_ID_RETENTION,
        "note": "Load the .keras file, load the two .npy files, then run the OOD "
                "cells. No retraining needed. Do not retrain to fit thresholds: "
                "a new run will not match the weights already shipped in the app.",
    }, f, indent=2)

print("Saved for later threshold fitting:")
for f in sorted(os.listdir(RESUME_DIR)):
    p = os.path.join(RESUME_DIR, f)
    print("   {:<38} {:>8.2f} MB".format(f, os.path.getsize(p) / 1048576))
print("")
print("Location:", RESUME_DIR)
print("")
print("Without these files, fitting margin and entropy later would require a full")
print("retrain, and the retrained weights would not match the ones in the app.")
